In [1]:
import json

import ollama
from typing import List
from typing import Literal
from pydantic import BaseModel
import networkx as nx

class Response(BaseModel):
    answ: Literal['Option1','Option2']
    

def testHalu(answer,correct,hallu):
    s='Consider the following two options:\n'
    s+='Option1: '+correct+'\n'
    s+='Option2: '+hallu+'\n'
    s+='Which one of the following options (Option 1 and Option 2) is more aligned in meeaning with \''+answer+'\' ?'
    res = ollama.generate(
    	#model="sunzhiyuan/suntray-vision",
        model='gemma3',
        #model="llama3.2-vision",
        prompt=s,
        format=Response.model_json_schema(),
        options= { "temperature": 0 }
    )
    dt=str(res['response'])
    if 'Option1' in dt:
        return False
    else:
        print('Halucination')
        print('Correct ANSW: '+correct)
        print('Hallu ANSW: '+hallu)
        print('Returned ANSW: '+answer)
        return True

def query(s):
    res = ollama.generate(
    	#model="sunzhiyuan/suntray-vision",
        model='gemma3',
        #model="llama3.2-vision",
        prompt=s,
        #format=ObjectDetectionResponse.model_json_schema(),
        options= { "temperature": 0 }
    )
    return  res['response']

In [8]:
with open('qa_data.json') as f:
    count=0
    tot=0
    nonHallucinated=[]
    hallucinated=[]
    for x in f:
        print(tot,count)
        #rint(x)
        data=json.loads(x)
        s='Given the following context:\n'+'-------\n'+data['knowledge']+'\n-------\n'+data['question']
        answer=query(s)
        if testHalu(answer,data['right_answer'],data['hallucinated_answer']):
            print(s)
            count+=1
            hallucinated.append(x)
        else:
            nonHallucinated.append(x)
        tot+=1
print('Hallucination rate:', count/tot)
    

0 0
1 0
2 0
3 0
4 0
5 0
6 0
Halucination
Correct ANSW: Crambidae
Hallu ANSW: The Indogrammodes genus of moths found in India has only one species.
Returned ANSW: According to the text, the genus of moth in the world’s seventh-largest country (India) that contains only one species is **Indogrammodes**.
Given the following context:
-------
Indogrammodes is a genus of moths of the Crambidae family. It contains only one species, Indogrammodes pectinicornalis, which is found in India.India, officially the Republic of India ("Bhārat Gaṇarājya"), is a country in South Asia. It is the seventh-largest country by area, the second-most populous country (with over 1.2 billion people), and the most populous democracy in the world.
-------
Which genus of moth in the world's seventh-largest country contains only one species?
7 1
8 1
9 1
10 1
Halucination
Correct ANSW: Jaime Meline
Hallu ANSW: Fast Cars, Danger, Fire and Knives includes guest appearances from Russell Simmons.
Returned ANSW: According 

In [1]:
hallucinated

NameError: name 'hallucinated' is not defined

In [7]:
class Response1(BaseModel):
    knoledge: str

def improve(s):
    res = ollama.generate(
    	#model="sunzhiyuan/suntray-vision",
        model='gemma3',
        #model="llama3.2-vision",
        prompt=s,
        #format=Response1.model_json_schema(),
        options= { "temperature": 0 }
    )
    return res['response']
    
count=0
with open('qa_data.json') as f:
    for x in f:
        print(count)
        count+=1
        data=json.loads(x)
        s=data['knowledge']
        
        s1='Given the following Knoledge text:\n------\n'+s+'\n------\nPlease rewrite the following knoledge text to avoid hallucination by preserving all the information.'
        s1+='Output only the rewritten text.'
        #print(s1)
        s2=improve(s1)
        #print(s2)
        s3='Given the following context:\n'+'-------\n'+s2+'\n-------\n'+data['question']
        answer=query(s3)
        if testHalu(answer,data['right_answer'],data['hallucinated_answer']):
            print(s3)
            print('original=',s)

0
1
2
3
4
5
6
7
Halucination
Correct ANSW: Badr Hari
Hallu ANSW: Badr Hari is a notorious kickboxer.
Returned ANSW: Badr Hari was once considered the best kickboxer in the world, however he has been involved in a number of controversies relating to his “unsportsmanlike conduct” in the sport and crimes of violence outside of the ring.
Given the following context:
-------
Fighters from around the world on the roster include Badr Hari, Peter Aerts, Peter Graham, Dewey Cooper, and Zabit Samedov. It was considered one of the biggest kickboxing and MMA promotions in the Middle East. Badr Hari (Arabic: بدر هاري‎ ; born 8 December 1984) is a Moroccan-Dutch super heavyweight kickboxer from Amsterdam, fighting out of Mike's Gym in Oostzaan. Hari has been a prominent figure in the world of kickboxing and was once considered the best kickboxer in the world, however he has been involved in a number of controversies relating to his "unsportsmanlike conduct" in the sport and crimes of violence outsid

KeyboardInterrupt: 

In [ ]:
testHalu('I am great', 'I am the best', 'I am bad')

In [14]:
testHalu('I am great',  'I am bad', 'I am the best',)

Halucination
Correct ANSW: I am bad
Halu ANSW: I am bad
Returned ANSW: I am great


True

In [60]:
import ollama
from typing import List
from typing import Literal
from pydantic import BaseModel
import networkx as nx

class Entity(BaseModel):
    id: int
    #startPosition: int
    #endPosition: int
    entityName: str

class Relation(BaseModel):
    EntityA : int
    EntityB : int
    relationNamFromAtoB : str


class ObjectDetectionResponse(BaseModel):
    nodes: List[Entity]
    edges: List[Relation]

def extractRDF(s1):
    s='Extract the entity relation graph from the following text with all the entities and all the relations: \"'+s1+'\"'
    res = ollama.generate(
    	#model="sunzhiyuan/suntray-vision",
        model='gemma3',
        #model="llama3.2-vision",
        prompt=s,
        format=ObjectDetectionResponse.model_json_schema(),
        options= { "temperature": 0 }
    )
    return  json.loads(res['response'])

In [64]:
s=json.loads(hallucinated[len(hallucinated)-3])
s1=s['knowledge']
s2=s['question']
print(s1)
print(s2)
print(extractRDF(s1))
print(extractRDF(s2))

"Semper Fidelis", written in 1888 by John Philip Sousa (The March King), is regarded as the official march of the United States Marine Corps. Because of his mastery of march composition, he is known as "The March King", or the "American March King" due to his British counterpart Kenneth J. Alford also being known by the former nickname.
Who is the British counterpart of the man who wrote "Semper Fidelis" in 1888?
{'nodes': [{'id': 1, 'entityName': 'Semper Fidelis'}, {'id': 2, 'entityName': 'John Philip Sousa'}, {'id': 3, 'entityName': 'The March King'}, {'id': 4, 'entityName': 'United States Marine Corps'}, {'id': 5, 'entityName': 'Kenneth J. Alford'}, {'id': 6, 'entityName': 'American March King'}], 'edges': [{'EntityA': 1, 'EntityB': 2, 'relationNamFromAtoB': 'written_by'}, {'EntityA': 1, 'EntityB': 4, 'relationNamFromAtoB': 'official_march_of'}, {'EntityA': 2, 'EntityB': 3, 'relationNamFromAtoB': 'known_as'}, {'EntityA': 2, 'EntityB': 5, 'relationNamFromAtoB': 'British counterpart o

In [68]:
import ollama
from typing import List
from typing import Literal
from pydantic import BaseModel
import networkx as nx


class RDFTuple(BaseModel):
    EntityA : str
    EntityB : str
    relationNamFromAtoB : str

class Outcome(BaseModel):
    KnoledgeRDFTuples: List[RDFTuple]
    QueryRDFTuples: List[RDFTuple]
    RDFTuplesBetweenKnoledgeQuery: List[RDFTuple]

def extractRDF(knoledge,query):
    s='Given the following:\n'
    s+='Knoledge Text:\n'
    s+=knoledge+'\n'
    s+='-----------\n'
    s+='Query Text:\n'
    s+=query+'\n'
    s+='-----------\n'
    s+='Extract the RDFTuples independently from the knowledge text and query text (with all the entities and all the relations), and the rlations from the knoledge and the query entities.'
    res = ollama.generate(
    	#model="sunzhiyuan/suntray-vision",
        model='gemma3',
        #model="llama3.2-vision",
        prompt=s,
        format=Outcome.model_json_schema(),
        options= { "temperature": 0 }
    )
    return  json.loads(res['response'])

In [69]:
s=json.loads(hallucinated[len(hallucinated)-3])
s1=s['knowledge']
s2=s['question']
print(s1)
print(s2)
print(extractRDF(s1,s2))

"Semper Fidelis", written in 1888 by John Philip Sousa (The March King), is regarded as the official march of the United States Marine Corps. Because of his mastery of march composition, he is known as "The March King", or the "American March King" due to his British counterpart Kenneth J. Alford also being known by the former nickname.
Who is the British counterpart of the man who wrote "Semper Fidelis" in 1888?
{'KnoledgeRDFTuples': [{'EntityA': 'Semper Fidelis', 'EntityB': 'John Philip Sousa', 'relationNamFromAtoB': 'written_by'}, {'EntityA': 'Semper Fidelis', 'EntityB': 'United States Marine Corps', 'relationNamFromAtoB': 'official_march_of'}, {'EntityA': 'John Philip Sousa', 'EntityB': 'The March King', 'relationNamFromAtoB': 'known_as'}, {'EntityA': 'John Philip Sousa', 'EntityB': 'American March King', 'relationNamFromAtoB': 'known_as'}, {'EntityA': 'Kenneth J. Alford', 'EntityB': 'The March King', 'relationNamFromAtoB': 'known_as'}], 'QueryRDFTuples': [{'EntityA': 'the man who 

In [79]:
ss=json.loads(hallucinated[len(hallucinated)-3])
print(ss)
s1=ss['knowledge']
s2=ss['question']
rdf=extractRDF(s1,s2)
s='Given the following context and query (with their associated RDF graphs):\n'
s+='Context:-------\n'+s1+'\n-------\n'
s+='RDFContext:-------\n'+str(rdf['KnoledgeRDFTuples'])+'\n-------\n'
s+='Query-------\n'+s2+'\n-------\n'
s+='RDFQuery:-------\n'+str(rdf['QueryRDFTuples'])+'\n-------\n'
s+='RDFTuplesBetweenKnoledgeQuery:-------\n'+str(rdf['RDFTuplesBetweenKnoledgeQuery'])+'\n-------\n'
s+='Please answer the query considering the information contained in the RDF graphs.'
answer=query(s)

{'knowledge': '"Semper Fidelis", written in 1888 by John Philip Sousa (The March King), is regarded as the official march of the United States Marine Corps. Because of his mastery of march composition, he is known as "The March King", or the "American March King" due to his British counterpart Kenneth J. Alford also being known by the former nickname.', 'question': 'Who is the British counterpart of the man who wrote "Semper Fidelis" in 1888?', 'right_answer': 'Kenneth J. Alford', 'hallucinated_answer': 'John Philip Sousa\'s British counterpart is Kenneth J. Alford- the "British March King".'}


In [80]:
answer

'Kenneth J. Alford'

In [81]:
def RDFQuery(ss):
    s1=ss['knowledge']
    s2=ss['question']
    rdf=extractRDF(s1,s2)
    s='Given the following context and query (with their associated RDF graphs):\n'
    s+='Context:-------\n'+s1+'\n-------\n'
    s+='RDFContext:-------\n'+str(rdf['KnoledgeRDFTuples'])+'\n-------\n'
    s+='Query-------\n'+s2+'\n-------\n'
    s+='RDFQuery:-------\n'+str(rdf['QueryRDFTuples'])+'\n-------\n'
    s+='RDFTuplesBetweenKnoledgeQuery:-------\n'+str(rdf['RDFTuplesBetweenKnoledgeQuery'])+'\n-------\n'
    s+='Please answer the query considering the information contained in the RDF graphs.'
    return query(s)

with open('qa_data.json') as f:
    count=0
    tot=0
    nonHallucinated=[]
    hallucinated=[]
    for x in f:
        print(tot,count)
        #rint(x)
        data=json.loads(x)
        answer=RDFQuery(data)
        if testHalu(answer,data['right_answer'],data['hallucinated_answer']):
            print(data)
            count+=1
            hallucinated.append(x)
        else:
            nonHallucinated.append(x)
        tot+=1
print('Hallucination rate:', count/tot)

0 0
1 0
2 0
Halucination
Correct ANSW: President Richard Nixon
Hallu ANSW: Allie Goertz wrote a song about Milhouse, a popular TV character, named after an influential political figure.
Returned ANSW: Allie Goertz wrote a song about Milhouse, who Matt Groening named after President Richard Nixon.
{'knowledge': 'Allison Beth "Allie" Goertz (born March 2, 1991) is an American musician. Goertz is known for her satirical songs based on various pop culture topics. Her videos are posted on YouTube under the name of Cossbysweater.Milhouse Mussolini van Houten is a fictional character featured in the animated television series "The Simpsons", voiced by Pamela Hayden, and created by Matt Groening who named the character after President Richard Nixon\'s middle name.', 'question': 'Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?', 'right_answer': 'President Richard Nixon', 'hallucinated_answer': 'Allie Goertz wrote a 

KeyboardInterrupt: 